# Module 02 — Operators

Arithmetic, comparison, logic, and the bit patterns that real devices report
their state with. About 25 minutes. Assumes module 01.

Predictions work as before: replace each `...`, and **silence means right**.

## 1. Precedence

Operators have a pecking order. From strongest to weakest:

| | |
|---|---|
| `**` | power |
| `-x` | sign |
| `* / // %` | multiply, divide, remainder |
| `+ -` | add, subtract |
| `< > <= >= == !=` | compare |
| `not` → `and` → `or` | logic |

`2 + 3 * 4` is 14, not 20. Two entries in that table hide a surprise, and both
are in the cell below.

In [ ]:
assert 2 + 3 * 4 == ...
assert 2**3**2 == ...
assert -(3**2) == ...

**`**` groups from the right.** `2 ** 3 ** 2` is `2 ** (3 ** 2)`, which is
`2 ** 9` — not `(2 ** 3) ** 2`. Every other arithmetic operator groups from the
left.

**`**` binds tighter than the minus sign.** `-3 ** 2` is `-(3 ** 2)`, so `-9`.
If you meant "minus three, squared", that is `(-3) ** 2`.

When in doubt, use brackets. They cost nothing and settle the argument.

## 2. The remainder of a negative number

`%` is straightforward for positive numbers. For negative ones, languages
disagree, and Python made a deliberate choice: **the result takes the sign of
the right-hand operand.**

In [ ]:
assert 17 % 5 == ...
assert -17 % 5 == ...

> **If you know C or Java:** there `-17 % 5` is `-2`, because the result takes
> the sign of the *left* operand. This is one of the few places where porting an
> arithmetic expression between the languages silently changes its meaning —
> and it bites hardest in exactly the place `%` is most used: wrapping an index
> around into a valid range.

## 3. Comparisons, and chaining them

Every comparison produces `True` or `False`. `==` compares values, `!=` is its
negation, and `<`, `>`, `<=`, `>=` do what they look like.

Python lets you **chain** them, and means it:

```python
-40 <= reading <= 85
```

reads as "reading is between −40 and 85 inclusive", and that is exactly what it
computes — both comparisons, with `reading` evaluated once.

In [ ]:
reading = 21.7
print(-40 <= reading <= 85)
print(5 == 5.0)
print("a" == "A")

Brackets turn the chain into something else entirely. **Predict both:**

In [ ]:
assert (3 > 2 > 1) is ...
assert ((3 > 2) > 1) is ...

The bracketed version computes `3 > 2` first, which is `True`; then asks
`True > 1`. And `True` counts as `1` in arithmetic, so that is `1 > 1` —
`False`.

> **If you know C or Java:** `3 > 2 > 1` there means the bracketed version, and
> it quietly evaluates to something useless. Chained comparison is one of the
> few pieces of Python syntax with no counterpart in either language, and it is
> worth using: `1 < x < 10` says what you mean.

## 4. `and`, `or`, `not` — and what they hand back

The truth tables hold no surprises:

| A | B | `A and B` | `A or B` |
|---|---|---|---|
| `True` | `True` | `True` | `True` |
| `True` | `False` | `False` | `True` |
| `False` | `True` | `False` | `True` |
| `False` | `False` | `False` | `False` |

The surprise is what the operators **return**. Not `True` or `False` — the
operand that settled the question.

In [ ]:
assert (0 or "empty") == ...
assert ("a" and "b") == ...

`0` is falsy, so `or` moves on and hands back `"empty"`. `"a"` is truthy, so
`and` has to look at the right side and hands back `"b"`.

That gives a common idiom for defaults:

```python
name = user_input or "unknown"     # falls back when user_input is empty
```

> **If you know C or Java:** `&&` and `||` there always produce a boolean, so
> this idiom does not exist and you reach for `?:` instead. Python's version is
> shorter but has a sharp edge: `count or 10` falls back for `count = 0` too,
> because zero is falsy. When zero is a legitimate value, test for `None`
> explicitly.

## 5. Short-circuit evaluation

Python stops as soon as the answer is settled. If the left side of `and` is
false, the right side is **never evaluated**.

This is not an optimisation you are lucky to get — it is a promise of the
language, and you are meant to write guards that depend on it.

In [ ]:
total = 0
count = 0

# The guard on the left means the division never happens.
print(count != 0 and total / count > 10)

Swap the two sides and it crashes. The order is not style; it is the whole
mechanism.

> **If you know C or Java:** `&&` and `||` behave identically. Nothing new here.

## 6. `==` against `is`

Two different questions, and confusing them is the most common mistake in the
language.

| | asks |
|---|---|
| `==` | do these have **the same value**? |
| `is` | are these **the same object** in memory? |

Two sheets of paper with `21.7` written on them are equal in value and are not
the same sheet.

In [ ]:
a = [1, 2]
b = [1, 2]

assert (a == b) is ...
assert (a is b) is ...

Same contents, different objects.

**Use `==` for values.** `is` has exactly one everyday use:

```python
if result is None:
```

That works because there is only ever one `None` in a running program, so
identity and equality coincide.

Do not be tempted to use `is` on numbers or short strings, even when it appears
to work. Whether two equal numbers happen to be the same object depends on how
Python was implemented and may change between versions — it is not something the
language promises you.

> **If you know Java:** this is `==` against `.equals()`, with the names
> swapped. Java's `==` compares references (Python's `is`), and `.equals()`
> compares values (Python's `==`). Reading Python with Java reflexes is exactly
> the wrong way round here, so it is worth slowing down for.

## 7. Bit masks

A device rarely reports its state in words. It reports **one byte**, where each
bit means one yes-or-no fact:

```
0 0 0 0 0 1 1 0
              └─ bit 0: measurement ready
            └─── bit 1: limit exceeded
          └───── bit 2: sensor fault
        └─────── bit 3: calibration due
```

That byte is the number 6. To ask about one bit you need a **mask**: a number
with a single bit set, in the position you care about. Then:

- `status & MASK` — **read** the bit. Everything else is masked away, so the
  result is zero when the bit is clear and non-zero when it is set.
- `status | MASK` — **set** it.
- `status & ~MASK` — **clear** it (`~` flips every bit of the mask).
- `status ^ MASK` — **toggle** it.

Write binary literals with `0b`, and read them back with `bin()`.

In [ ]:
READY, LIMIT, FAULT, CALIBRATE = 0b0001, 0b0010, 0b0100, 0b1000
status = 0b0110

# What is left after masking away everything but the LIMIT bit?
assert (status & LIMIT) == ...

# And the READY bit, which is not set?
assert (status & READY) == ...

Note that `status & LIMIT` gives you a **number**, not `True`. It is the value
of that bit in its own column — 2, not 1. When you want an answer to the
question "is it set", wrap it: `bool(status & LIMIT)`.

In [ ]:
READY, LIMIT, FAULT, CALIBRATE = 0b0001, 0b0010, 0b0100, 0b1000
status = 0b0110

print(bool(status & LIMIT))
print(bin(status | READY))  # set READY
print(bin(status & ~FAULT))  # clear FAULT

> **If you know C:** `&`, `|`, `^`, `~`, `<<` and `>>` are the same operators
> doing the same job, and this idiom is the same one you already use on
> hardware registers. The one difference is that Python integers have no fixed
> width, so `~` on a positive number gives you a negative one rather than a
> pattern of a known size.

**Where you meet this:** GPIO pins on a microcontroller, status registers in
datasheets, file permissions (`chmod 755`), flags in network protocols.
Anywhere many yes-or-no facts have to fit into one number.

---

## Done

On to `exercises/`, then `uv run pytest 02_operators`.